In [3]:
import time
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
    RunnableBranch
)
from IPython.display import display, Markdown, HTML

MODEL = "llama3.2:3b"

llm = ChatOllama(model = MODEL, temperature= 0.1)

response = llm.invoke("Say Hello in one word")
print(f"Response : {response.content}")


Response : Hello.


---

## 2. Extending the Chain — Adding More Components

Each additional component in the pipe adds a processing step. The chain remains a single Runnable.

```
chain = prompt | model | parser | post_processor | formatter
```

### Experiment 2A: Extending with Post-Processing

In [5]:
# Add a post-processing step using RunnableLambda
def add_disclaimer(text):
    return f"{text} --- Generated by model with LCEL"

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are technical assistant"),
    ("human", "Explain {question}")
])
chain = prompt | llm | StrOutputParser() | RunnableLambda(add_disclaimer)

result = chain.invoke({"question" : "What is Pyhton?"})
print(result)

**What is Python?**

Python is a high-level, interpreted programming language that is widely used for various purposes such as web development, scientific computing, data analysis, artificial intelligence, and more. Created in the late 1980s by Guido van Rossum, Python is known for its simplicity, readability, and ease of use.

**Key Features of Python:**

1. **Easy to Learn**: Python has a simple syntax and is relatively easy to learn, making it a great language for beginners.
2. **High-Level Language**: Python is a high-level language, meaning it abstracts away many low-level details, allowing developers to focus on the logic of their program without worrying about memory management, etc.
3. **Interpreted Language**: Python code is interpreted line by line, making it easy to write and test programs quickly.
4. **Object-Oriented**: Python supports object-oriented programming (OOP) concepts such as classes, objects, inheritance, polymorphism, and encapsulation.
5. **Large Standard Libr

## 3. LCEL Primitive 1: RunnablePassthrough — Pass Data Through

### Experiment 3C: assign() in a Real Chain — Adding Context

In [12]:
 # Practical use: enrich input with LLM-generated context before the main prompt

# Step 1: Generate keywords from the question
keyword_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Extract 3 keywords from this question.No extra text, only heading."),
        ("human", "{question}")
    ])
    | llm
    | StrOutputParser()
)


enrich_chain = RunnablePassthrough.assign(
    keywords = keyword_chain    
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question. Focus on these keyword : {keywords}"),
    ("human", "{question}")
])

final_chain = enrich_chain | final_prompt | llm | StrOutputParser()

result = final_chain.invoke({
    "question": "What is Python"})
print("Result :")
display(Markdown(result))

Result :


Python is a high-level, interpreted programming language that is widely used for various purposes such as web development, scientific computing, data analysis, artificial intelligence, and more.

Created in the late 1980s by Guido van Rossum, Python was designed to be easy to learn and use, with a focus on code readability and simplicity. It has since become one of the most popular programming languages in the world, known for its versatility, flexibility, and large community of developers who contribute to its ecosystem.

Some key features of Python include:

1. **Easy to learn**: Python has a simple syntax and is relatively easy to read and write, making it a great language for beginners.
2. **High-level language**: Python abstracts away many low-level details, allowing developers to focus on the logic of their program without worrying about memory management or other details.
3. **Interpreted language**: Python code is interpreted line-by-line, which means that it can be executed immediately without the need for compilation.
4. **Object-oriented**: Python supports object-oriented programming (OOP) concepts such as classes, objects, inheritance, and polymorphism.
5. **Large standard library**: Python has a vast collection of libraries and modules that make it easy to perform various tasks, such as file I/O, networking, and data analysis.

Python is widely used in many areas, including:

1. **Web development**: Python is used in web frameworks such as Django and Flask to build scalable and efficient web applications.
2. **Data science and analytics**: Python is a popular choice for data analysis, machine learning, and scientific computing, thanks to libraries like NumPy, pandas, and scikit-learn.
3. **Artificial intelligence and machine learning**: Python is used in AI and ML applications, such as natural language processing, computer vision, and predictive modeling.
4. **Automation and scripting**: Python is often used for automating tasks, such as data scraping, file management, and system administration.

Overall, Python is a powerful and versatile programming language that has become an essential tool for many developers and professionals in various fields.

## 4. LCEL Primitive 2: RunnableParallel — Execute Branches Simultaneously

### Experiment 4A: Basic RunnableParallel

In [ ]:
# Two branches process the same input in parallel
summary_chain = (
    ChatPromptTemplate([
        ("system", "Summarize in one sentence"),
        ("human","{topic}")
    ])
    | llm | StrOutputParser()
)

achievement_chain = (
    ChatPromptTemplate([
        ("system", "List 3 achievements. Comma, seperated value"),
        ("human", "{topic}")
    ])
    | llm | StrOutputParser()
)

# RunnableParallel — both execute at the same time
full_chain = RunnableParallel(
    summary = summary_chain,
    achievements = achievement_chain
)

result  = full_chain.invoke({"topic" : "MS Dhoni"})
print(f"Summary : {result['summary']}")
print(f"Achievements : {result['achievements']}")

# Result is a dictionary, so used ['summary']
# result = {"summary":"MS Dhoni is ... ", "achievements":"...."}

Summary : MS Dhoni is a former Indian international cricketer and former captain of the India national team, widely regarded as one of the greatest wicket-keepers in cricket history.
Achievements : Here are three achievements of MS Dhoni:

Ranji Trophy winner (2004-05), Indian Premier League (IPL) champion (2010), ICC World Twenty20 winner (2007)


### Experiment 4C: RunnableParallel + RunnablePassthrough (The RAG Pattern)

In [21]:
# Simulated RAG pattern — no vector DB needed, just the LCEL structure
# Simulate a retriever with a function

knowledge = {
    "lcel": "LCEL is LangChain's declarative composition framework using the pipe operator.",
    "runnable": "The Runnable protocol provides invoke(), stream(), and batch() on every component.",
    "passthrough": "RunnablePassthrough forwards input unchanged, assign() adds new keys.",
    "parallel": "RunnableParallel executes multiple branches simultaneously.",
}

def fake_retriever(query:str)->str:
    query_lower = query.lower()
    mathed = []
    for key,doc in knowledge.items():
        if key in query_lower:
            mathed.append(doc)
    return " ".join(mathed) if mathed else "No relevant documents found." 

rag_prompt = ChatPromptTemplate.from_messages({
    ("system", "Answer the questions based on this context:\n\n{context}\n\n"
     "Summarize in one sentene"),
    ("human", "Explain {question}")
})

rag_chain = (
    {"context" : RunnableLambda(lambda x: fake_retriever(x['question'])),
     "question" : RunnablePassthrough() | RunnableLambda(lambda x : x['question'])}
    | rag_prompt
    | llm
    | StrOutputParser()
)

questions = [
    "What is LCEL?",
    "How does RunnableParallel work?",
    "What is the weather today?",  # Not in knowledge base    
]

for q in questions:
    print(f"Q : {q}")
    result = rag_chain.invoke({"question":q})
    print(f"A : {result}\n")

Q : What is LCEL?
A : LCEL (LangChain's Declarative Composition Framework) is a programming framework that uses the pipe operator to enable declarative composition of data processing and workflow automation.

Q : How does RunnableParallel work?
A : RunnableParallel works by executing multiple branches of a program concurrently, utilizing the invoke() method to run each branch independently, allowing for parallel execution and potentially improving performance.

Q : What is the weather today?
A : I don't have any information about the current weather, as no relevant documents were provided.



## 5. LCEL Primitive 3: RunnableLambda — Custom Functions as Runnables

### Experiment 5C: The @chain Decorator — Cleaner Syntax

In [22]:
from langchain_core.runnables import chain as chain_decorator

# The @chain decorator turns a function into a Runnable
# Cleaner than wrapping with RunnableLambda
@chain_decorator
def analyze_and_reuturn(input_dict:dict)->str:
    question = input_dict["question"]

    # Step 1 : Clasify the question type
    classify_chain = (
        ChatPromptTemplate.from_messages([
            ("system", "Classify this as 'technical' or 'general'. One word only."),
            ("human", "{q}")
        ])
        | llm
        | StrOutputParser()
    )

    category = classify_chain.invoke({"q" : question}).strip().lower()

    # Step 2: Answer based on classification
    if "technical" in category:
        style = "a precise software engineer with code examples"
    else:
        style = "a friendly teacher using simple analogies"

    answer_chain = (
        ChatPromptTemplate.from_messages([
            ("system", "You are {style}. Keep answer 2 or 3 sentences only."),
            ("human", "{q}")
        ])
        | llm
        | StrOutputParser()
    ) 

    answer = answer_chain.invoke({"style" : style, "q": question})
    return f"[{category.upper()}]  {answer}"

result = analyze_and_reuturn.invoke({"question": "How does the pipe operator work in LCEL?"})
print(f"Result of technical : {result}")

result1 = analyze_and_reuturn.invoke({"question" : "What is Rainbow?"})
print(f"Result of general : {result1}")

Result of technical : [TECHNICAL]  In Lisp Common Lisp (LCL), the pipe operator `|>` is used to create a new function that applies a given function to each element of its input list. Here's an example:
```lisp
(defun double (x) (* x 2))
(defparameter *list* '(1 2 3))

(defparameter result (reduce '|double| *list*) ; equivalent to mapcar 'double *list*
                         :initial-value nil)
```
This creates a new list with the doubled values of each element in `*list*`.
Result of general : [GENERAL]  Think of a rainbow like a big, colorful smile in the sky! It's created when sunlight passes through tiny drops of water in the air, like after a rain shower. This makes all the different colors shine together, creating that beautiful arc we see.


## 6. LCEL Primitive 4: RunnableBranch — Conditional Routing

### Experiment 6A: Basic Conditional Routing

In [ ]:
# Different chain with different input types

technical_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a Python engineer. Use technical terminology in 2 sentences."),
        ("human", "{question}")
    ])
    | llm
    | StrOutputParser()
    | RunnableLambda(lambda x: f"[TECHNICAL] {x}")
)

creative_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a creative storytellar. Be concise, use metaphors and vivid language."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[CREATIVE] {x}")
)

general_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI assistant. Keep 2 or 3 sentences."),
        ("human", "{question}")
    ])
    | llm | StrOutputParser()
    | RunnableLambda(lambda x: f"[GENERAL] {x}")
)

# Route based on keywords in the question
router = RunnableBranch(
    (lambda x: any(w in x["question"].lower() for w in ["lambda", "python", "code"]),
     technical_chain),
    (lambda x:any(w in x["question"].lower() for w in ["story", "imagine", "poem"]),
     creative_chain),
    general_chain
)
test_questions =[
    "What is lambda in Python?",
    "Imagine you are a poet.Tell me a poem about moon in 2 lines",
    "Who is captain of Indian cricket team in T20 format?"
]

for q in test_questions:
    print("=" * 50)
    print(f"Q : {q}")
    print("=" * 50)
    result = router.invoke({"question":q})
    print(result)

Q : What is lambda in Python?
[TECHNICAL] In Python, `lambda` refers to an anonymous function, which is a small, single-purpose function that can be defined inline within a larger expression. The syntax for defining a lambda function is `lambda arguments: expression`, where `arguments` are the input parameters and `expression` is the code that gets executed when the lambda function is called.
Q : Imagine you are a poet.Tell me a poem about moon in 2 lines
[CREATIVE] "Luna's silvery crescent smile,
Dances on the midnight sky, a gentle, ethereal guile."
Q : Who is captain of Indian cricket team in T20 format?
[GENERAL] As of my knowledge cutoff in 2023, Rohit Sharma is the captain of the Indian national cricket team in the T20 International (T20I) format. He has been leading the team since 2017 and has been instrumental in their success in various tournaments. However, please note that team captains can change over time due to various reasons.
